In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [108]:
# Define the nodes
nodes = [
    {'label': 'Node 1'},
    {'label': 'Node 2'},
    {'label': 'Node 3'},
    {'label': 'Node 4'},
    {'label': 'Node 5'},
    {'label': 'Node 6'},
    {'label': 'Node 7'}, 
    {'label': 'Node 8'}
]

# Define the links
links = [
    {'source': 0, 'target': 1, 'value': 10},  # Flow from Node 1 to Node 2 with a value of 10
    {'source': 1, 'target': 2, 'value': 5},   # Flow from Node 2 to Node 3 with a value of 5
    {'source': 2, 'target': 3, 'value': 1},   # Flow from Node 3 to Node 1 with a value of 1
    {'source': 4, 'target': 2, 'value': 2},   # Flow from Node 4 to Node 1 with a value of 2
    {'source': 5, 'target': 1, 'value': 8},   # Flow from Node 5 to Node 1 with a value of 8
    {'source': 6, 'target': 3, 'value': 3},   # Flow from Node 6 to Node 1 with a value of 3
    {'source': 6, 'target': 1, 'value': 7},   # Flow from Node 7 to Node 1 with a value of 7
    {'source': 1, 'target': 7, 'value': 4},   # Flow from Node 8 to Node 1 with a value of 4
    {'source': 7, 'target': 3, 'value': 6},   # Flow from Node 9 to Node 1 with a value of 6
]
type(links[0])
print(len(nodes))

8


In [116]:
# Create the figure and subplot
fig = make_subplots(rows=1, cols=1)

# Add the Sankey trace
fig.add_trace(go.Sankey(
    node=dict(
        pad=10*len(nodes),  # Adjust the padding between nodes
        thickness=20,  # Adjust the thickness of nodes
        line=dict(color='black', width=1),  # Define the color and width of node lines
        label=[node['label'] for node in nodes],  # Extract node labels from the 'nodes' list
    ),
    link=dict(
        source=[link['source'] for link in links],  # Extract source indices from the 'links' list
        target=[link['target'] for link in links],  # Extract target indices from the 'links' list
        value=[link['value'] for link in links],  # Extract values from the 'links' list
    )
))

# Update the layout
fig.update_layout(title_text='Sankey Diagram')

# Show the plot
fig.show()


In [62]:
from sklearn.cluster import KMeans
import numpy as np

[[7. 2. 8. 6. 4. 0. 1. 3. 5. 9.]
 [7. 2. 8. 6. 4. 0. 1. 3. 5. 1.]
 [7. 2. 0. 6. 4. 0. 1. 3. 5. 1.]
 [1. 4. 3. 6. 0. 3. 1. 2. 5. 1.]
 [1. 3. 2. 4. 5. 2. 1. 0. 4. 1.]
 [1. 0. 4. 2. 0. 4. 1. 3. 2. 1.]
 [1. 2. 0. 0. 2. 0. 1. 3. 0. 1.]
 [1. 2. 2. 0. 2. 0. 1. 0. 0. 1.]
 [1. 0. 0. 0. 0. 0. 1. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]


In [130]:
import numpy as np
from sklearn.cluster import KMeans

def detect_label_switching(
        data : np.ndarray,
        num_clusters_range: range
        ) -> np.ndarray:
    """
    Detects label switching in KMeans clustering from one iteration to the next.

    Parameters
    ----------
    data : numpy.ndarray
        One-dimensional array of data points to cluster.
    num_clusters_range : range
        Range of number of clusters to try.
    
    Returns
    -------
    numpy.ndarray
        Array of labels for each data point for each iteration of KMeans clustering.
    """
    prev_labels = None  # Initialize previous labels as None
    km_labels = np.zeros((data.shape[0],data.shape[0]), dtype=int)
    km_labels_memorize = np.zeros((data.shape[0],data.shape[0]), dtype=int)

    for num_clusters in num_clusters_range:
        kmeans = KMeans(n_clusters=data.shape[0] - num_clusters, random_state=0)
        kmeans.fit(data.reshape(-1, 1))
        labels = kmeans.labels_
        km_labels[num_clusters] = labels

        if prev_labels is not None:
            # Compare current labels with previous labels
            switched_indices = np.where(labels != prev_labels)[0]
            if len(switched_indices) > 0:
                print(f"Label switching detected at iteration {num_clusters - 1} to {num_clusters}:")
                for index in switched_indices:
                    prev_label = prev_labels[index]
                    curr_label = labels[index]
                    print(f"Data point at index {index}: switched from label {prev_label} to label {curr_label}")
        if num_clusters == 0:
            km_labels_memorize[num_clusters] = labels
        else:
            km_labels_memorize[num_clusters] = km_labels_memorize[num_clusters - 1]
            for i, index in enumerate(switched_indices):
                all_index = np.where(km_labels[num_clusters] == labels[index])[0]
                km_labels_memorize[num_clusters][all_index] = int(np.max(km_labels_memorize[num_clusters - 1])) + i + 1
        prev_labels = labels
    return km_labels, km_labels_memorize

# Example usage:
data = np.array([0.59830198, 0.7101328 , 0.25210179, 0.90204906, 0.79986231, 0.25293537]) #0.21397742, 0.50635925, 0.67623288, 0.74856172, 
num_clusters_range = range(data.shape[0])  # Example range of number of clusters

result, result_new_labels = detect_label_switching(data, num_clusters_range)


Label switching detected at iteration 0 to 1:
Data point at index 2: switched from label 5 to label 1
Label switching detected at iteration 1 to 2:
Data point at index 0: switched from label 2 to label 0
Data point at index 1: switched from label 3 to label 2
Data point at index 3: switched from label 4 to label 3
Data point at index 4: switched from label 0 to label 2
Label switching detected at iteration 2 to 3:
Data point at index 0: switched from label 0 to label 2
Data point at index 3: switched from label 3 to label 0
Data point at index 4: switched from label 2 to label 0
Label switching detected at iteration 3 to 4:
Data point at index 0: switched from label 2 to label 0
Data point at index 1: switched from label 2 to label 0
Label switching detected at iteration 4 to 5:
Data point at index 2: switched from label 1 to label 0
Data point at index 5: switched from label 1 to label 0


In [131]:
print(result)
print(result_new_labels)

[[2 3 5 4 0 1]
 [2 3 1 4 0 1]
 [0 2 1 3 2 1]
 [2 2 1 0 0 1]
 [0 0 1 0 0 1]
 [0 0 0 0 0 0]]
[[ 2  3  5  4  0  1]
 [ 2  3  6  4  0  6]
 [ 7 10  6  9 10  6]
 [11 11  6 13 13  6]
 [15 15  6 15 15  6]
 [17 17 17 17 17 17]]


In [84]:
import numpy as np
import plotly.graph_objects as go
from sklearn.cluster import KMeans

def visualize_label_switching(data, num_clusters_range):
    prev_labels = None  # Initialize previous labels as None
    links = []

    for num_clusters in num_clusters_range:
        kmeans = KMeans(n_clusters=num_clusters)
        kmeans.fit(data.reshape(-1, 1))
        labels = kmeans.labels_

        if prev_labels is not None:
            # Compare current labels with previous labels
            switched_indices = np.where(labels != prev_labels)[0]
            if len(switched_indices) > 0:
                for index in switched_indices:
                    prev_label = prev_labels[index]
                    curr_label = labels[index]
                    links.append({"source": prev_label, "target": curr_label, "value": 1})

        prev_labels = labels

    # Create nodes
    nodes = [{"label": label} for label in np.unique(prev_labels)]

    # Create Sankey diagram
    fig = go.Figure(data=[go.Sankey(node=dict(label=node["label"]), link=dict(source=links["source"], target=links["target"], value=links["value"])) for node in nodes], layout_title_text="Label Switching Sankey Diagram")

    # Show the Sankey diagram
    fig.show()

# Example usage:
data = np.array([2, 4, 6, 10, 12, 15, 18, 20, 25, 30])
num_clusters_range = range(2, 6)  # Example range of number of clusters

visualize_label_switching(data, num_clusters_range)


TypeError: list indices must be integers or slices, not str